In [2]:
import os
import sys
print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\research


In [3]:
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\src")
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01")

In [4]:
%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Regression_01\\research'

In [5]:
os.chdir("../")

In [6]:
%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Regression_01'

In [7]:
import box
print(box.__version__)

7.4.1


In [8]:
# entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir : Path
    STATUS_FILE : Path
    unzip_data_dir : Path
    transformed_train_path : Path
    transformed_test_path : Path
    preprocessor_path : Path

In [9]:
from Regression_01.entity.config_entity import DataTransformationConfig
from Regression_01.utils.common import read_yaml, create_directories
from Regression_01.constant import CONFIG_FILE_PATH

In [10]:
# configuration manager

from Regression_01.config.configuration import ConfigurationManager

def get_data_transformation_config(self) -> DataTransformationConfig:

    config = self.config.data_transformation

    create_directories([config.root_dir])

    data_transformation_config = DataTransformationConfig(

        root_dir=Path(config.root_dir),

        STATUS_FILE=Path(config.STATUS_FILE), 

        unzip_data_dir=Path(config.unzip_data_dir),

        transformed_train_path=Path(config.transformed_train_path),

        transformed_test_path=Path(config.transformed_test_path),

        preprocessor_path=Path(config.preprocessor_path)

    )

    return data_transformation_config

In [11]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

from Regression_01.entity.config_entity import DataTransformationConfig
from Regression_01.logging import logger

[2026-07-30 01:12:37,822: INFO: utils: NumExpr defaulting to 4 threads.]


In [12]:
def binary_encoding(X):
                            #calling binary encoding first because Pipeline expects an estimator.

                            #FunctionTransformer converts our function into an estimator.
    X = X.copy()            # copy is used to prevent changing the original dataframe accidentally.

    mapping = {
        "yes": 1,
        "no": 0
    }

    return X.replace(mapping).infer_objects(copy=False)  

In [13]:
# component

class DataTransformation:

    def __init__(self, config: DataTransformationConfig):
        self.config = config


    def initiate_data_transformation(self):

        logger.info("***** Data Transformation Started *****")

        # Check Validation Status


        with open(self.config.STATUS_FILE,"r") as f:

            status = f.read().split(" ")[-1]

        if status != "True":
            raise Exception("Data Validation Failed")

        logger.info("Validation Successful")


        # -----------------------------
        # Read Dataset
        # -----------------------------

        df = pd.read_csv(self.config.unzip_data_dir)

        logger.info(f"Dataset Shape : {df.shape}")


        # -----------------------------
        # Split Features and Target
        # -----------------------------

        X = df.drop(columns=["price"],axis=1)

        y = df["price"]


        # -----------------------------
        # Train Test Split
        # -----------------------------

        X_train,X_test,y_train,y_test = train_test_split(

            X,
            y,
            test_size=0.20,
            random_state=42

        )

        logger.info("Train Test Split Completed")


        # -----------------------------
        # Columns
        # -----------------------------

        binary_columns = [

            "mainroad",
            "guestroom",
            "basement",
            "hotwaterheating",
            "airconditioning",
            "prefarea"

        ]

        categorical_column = [

            "furnishingstatus"

        ]


        # -----------------------------
        # Pipelines
        # -----------------------------

        binary_pipeline = Pipeline(

            steps=[

                ("binary",FunctionTransformer(binary_encoding))

            ]

        )


        categorical_pipeline = Pipeline(

            steps=[

                ("onehot",OneHotEncoder(handle_unknown="ignore"))

            ]

        )


        # -----------------------------
        # Column Transformer
        # -----------------------------

        preprocessor = ColumnTransformer(

            transformers=[

                ("binary",binary_pipeline,binary_columns),

                ("categorical",categorical_pipeline,categorical_column)

            ],

            remainder="passthrough"

        )


        logger.info("Applying Preprocessing")


        X_train = preprocessor.fit_transform(X_train)

        X_test = preprocessor.transform(X_test)


        logger.info("Preprocessing Completed")
        # -----------------------------
        # Save Preprocessor
        # -----------------------------

        joblib.dump(preprocessor, self.config.preprocessor_path)

        logger.info("Preprocessor Saved")


        # -----------------------------
        # Combine Features and Target
        # -----------------------------

        train_arr = np.c_[X_train, np.array(y_train)]

        test_arr = np.c_[X_test, np.array(y_test)]


        # -----------------------------
        # Save Train and Test Arrays
        # -----------------------------

        np.save(self.config.transformed_train_path, train_arr)

        np.save(self.config.transformed_test_path, test_arr)

        logger.info("Transformed Data Saved")


        return (
        self.config.transformed_train_path,
        self.config.transformed_test_path,
        self.config.preprocessor_path

    )


In [14]:
# Pipeline

STAGE_NAME = "Data Transformation Stage"


class DataTransformationTrainingPipeline:

    def __init__(self):
        pass

    def main(self):
        config = ConfigurationManager()

        data_transformation_config = (
            config.get_data_transformation_config()
        )

        data_transformation = DataTransformation(
            config=data_transformation_config
        )

        data_transformation.initiate_data_transformation()

In [15]:
obj = DataTransformationTrainingPipeline()

obj.main()

[2026-07-30 01:12:45,282: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\config\config.yaml loaded successfully]
[2026-07-30 01:12:45,288: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\params.yaml loaded successfully]
[2026-07-30 01:12:45,295: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\schema.yaml loaded successfully]
[2026-07-30 01:12:45,299: INFO: common: created directory at artifacts]
[2026-07-30 01:12:45,301: INFO: common: created directory at artifacts/data_transformation]
[2026-07-30 01:12:45,302: INFO: 3980954841: ***** Data Transformation Started *****]
[2026-07-30 01:12:45,306: INFO: 3980954841: Validation Successful]
[2026-07-30 01:12:45,318: INFO: 3980954841: Dataset Shape : (545, 13)]
[2026-07-30 01:12:45,329: INFO: 3980954841: Train Test Split Completed]
[2026-07-30 01:12:45,331: INFO: 3980954841: Applying Preprocessing]
[2026-07-30 01:12:45,356: INFO: 39

C:\Users\Greesha Vaishnavi\AppData\Local\Temp\ipykernel_5932\4019714624.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return X.replace(mapping).infer_objects(copy=False)
C:\Users\Greesha Vaishnavi\AppData\Local\Temp\ipykernel_5932\4019714624.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return X.replace(mapping).infer_objects(copy=False)
